# Wikipedia Summarizer/QA code

In [ ]:
### imports
import requests
import openai
import os
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

### Web-scraping function

In [ ]:
def read_webpage(URL):
    """
        Takes a wikipedia article url and grabs/returns the body text from the article 
        @Args:
            URL: the url for the website that will be scraped 
    """

    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.149 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://example.com' 
    }

    page = requests.get(url=URL, headers=headers)

    # Check if the request was successful (status code 200)
    if page.status_code == 200:
        print("Page fetched successfully.")


        soup = BeautifulSoup(page.content, "html.parser")
        # Find all elements with a specific tag and class
        # job_elements = soup.find_all("div", class_="card-content")
        article_text = []

        title = soup.find("h1").text.strip()
        for p in soup.find_all("p"):
            article_text.append(p.text.strip())
        print(f"Wiki Title: {title}")
        return article_text
    else:
        print(f"Failed to fetch page. Status code: {page.status_code}")
        return -1


### AI interface/prompter

* **you will need a OpenAI API key**

In [ ]:
def summarize_QA_GPT(html_data, QA=None):
    """
        submits both wikipage text and questions to OpenAI for summarization or question/answering
        @Args:
            html_data: wikipage's text data
            QA: the potential list of questions for the AI, also determines whether to summarize or answer questions
    """

    client = openai.OpenAI(api_key= os.environ["OPENAI_RIVAL_IO_API_KEY"])
    
    if QA is None:  # Summarize Text
        prompt_text = (
            "Analyze the following text content and summarize its content: "
            f"\n\n{html_data}"
        )

        response = client.chat.completions.create(
            model="gpt-4o", # GPT-4o is excellent at parsing raw HTML
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes data from text in bullet-point format."},
                {"role": "user", "content": prompt_text}
            ],
            temperature=0.1,
        )
        content = response.choices[0].message.content
        # Display as formatted markdown or return plain text

        display(Markdown(content))
        return content  # Also return the raw text for further use if needed
    else:   # get answers about text
        questions = "\n\n".join(QA)
        prompt_text = (
            "Answer the following questions based on the Text: "
            f"{questions} \n"
            f"Text: \n\n{html_data}"
        )
        response = client.chat.completions.create(
            model="gpt-4o", # GPT-4o is excellent at parsing raw HTML
            messages=[
                {"role": "system", "content": "You are a helpful assistant that answers questions based on data from text."},
                {"role": "user", "content": prompt_text}
            ],
            temperature=0.1,
        )
        content = response.choices[0].message.content
        # Display as formatted markdown or return plain text

        display(Markdown(content))
        return content  # Also return the raw text for further use if needed

### Main function call

In [ ]:
def cortexone_handler(event):
    """
        Takes a dictionary with a wikipedia url and questions and returns a AI summary or answers
        Args:
            event: dictionary with fields 'url' and 'questions'
    """
    
    html = read_webpage(event.get('url', ""))
    if html == -1:
        return {'statusCode': 404, 'body': f"Failed to fetch page."}
    
    questions = event.get('questions', None)
    
    return {'statusCode': 200, 'body': summarize_QA_GPT("\n".join(html), QA=questions)}
    